In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv('du_lieu_da_can_bang_downsampling.csv')

X = df.drop('label', axis=1)
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Tach test set rieng de danh gia cuoi cung
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Tach validation set de dung early stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

model_xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    gamma=0.1,
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    early_stopping_rounds=30
)

print("Bat dau huan luyen XGBoost voi early stopping")
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_pred_train = model_xgb.predict(X_train)
y_pred_val = model_xgb.predict(X_val)
y_pred_test = model_xgb.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
val_acc = accuracy_score(y_val, y_pred_val)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Validation Accuracy: {val_acc * 100:.2f}%")
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Overfitting gap (train - test): {(train_acc - test_acc) * 100:.2f}%")

print("\nBang diem chi tiet tren test set:")
target_names = label_encoder.inverse_transform(range(len(label_encoder.classes_)))
print(
    classification_report(
        y_test,
        y_pred_test,
        labels=list(range(len(label_encoder.classes_))),
        target_names=target_names,
        zero_division=0
    )
)

model_xgb.save_model('mo_hinh_xgboost_cu_chi.json')
joblib.dump(label_encoder, 'bo_giai_ma_nhan_xgboost.pkl')
print("Da luu 'mo_hinh_xgboost_cu_chi.json' va 'bo_giai_ma_nhan_xgboost.pkl'.")

Bat dau huan luyen XGBoost voi early stopping
Train Accuracy: 100.00%
Validation Accuracy: 99.42%
Test Accuracy: 99.59%
Overfitting gap (train - test): 0.41%

Bang diem chi tiet tren test set:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       189
           B       0.99      1.00      1.00       189
           C       0.99      0.99      0.99       190
           D       1.00      1.00      1.00       190
           E       1.00      0.98      0.99       190
           F       0.99      1.00      1.00       189
           G       0.99      0.98      0.99       190
           H       1.00      1.00      1.00       190
           I       1.00      1.00      1.00       189
           J       0.99      1.00      1.00       189
           K       1.00      1.00      1.00       190
           L       1.00      1.00      1.00       189
           M       0.99      0.99      0.99       190
           N       0.99      0.99      0.99       

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv('du_lieu_da_can_bang_downsampling.csv')

X = df.drop('label', axis=1)
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=16,
    min_samples_split=6,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

print("Bat dau huan luyen RandomForest")
model_rf.fit(X_train, y_train)

y_pred_train = model_rf.predict(X_train)
y_pred_test = model_rf.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"OOB Score: {model_rf.oob_score_ * 100:.2f}%")
print(f"Overfitting gap (train - test): {(train_acc - test_acc) * 100:.2f}%")

cv_scores = cross_val_score(
    model_rf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1
)
print(f"5-fold CV Accuracy: {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)")

print("\nBang diem chi tiet tren test set:")
target_names = label_encoder.inverse_transform(range(len(label_encoder.classes_)))
print(
    classification_report(
        y_test,
        y_pred_test,
        labels=list(range(len(label_encoder.classes_))),
        target_names=target_names,
        zero_division=0
    )
)

joblib.dump(model_rf, 'mo_hinh_randomforest_cu_chi.pkl')
joblib.dump(label_encoder, 'bo_giai_ma_nhan_randomforest.pkl')
print("Da luu 'mo_hinh_randomforest_cu_chi.pkl' va 'bo_giai_ma_nhan_randomforest.pkl'.")

Bat dau huan luyen RandomForest
Train Accuracy: 100.00%
Test Accuracy: 99.82%
OOB Score: 99.78%
Overfitting gap (train - test): 0.18%
5-fold CV Accuracy: 99.68% (+/- 0.12%)

Bang diem chi tiet tren test set:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       189
           B       1.00      1.00      1.00       189
           C       1.00      0.99      1.00       190
           D       1.00      1.00      1.00       190
           E       1.00      0.98      0.99       190
           F       1.00      1.00      1.00       189
           G       1.00      0.99      1.00       190
           H       1.00      1.00      1.00       190
           I       1.00      1.00      1.00       189
           J       1.00      1.00      1.00       189
           K       1.00      1.00      1.00       190
           L       1.00      1.00      1.00       189
           M       1.00      1.00      1.00       190
           N       1.00      0.99  